In [13]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# Each block is (num, den): two coeff arrays, highest power of s first.
def series(g1, g2):                          # cascade: multiply
    return (np.polymul(g1[0], g2[0]), np.polymul(g1[1], g2[1]))

def feedback(g, h):                          # negative feedback: G/(1+GH)
    n = np.polymul(g[0], h[1])
    d = np.polyadd(np.polymul(g[1], h[1]), np.polymul(g[0], h[0]))
    return (n, d)

Km, tau = 4.0, 0.25
G = (np.array([Km]), np.array([tau, 1.0, 0.0]))   # Km / (tau s^2 + s)
print("G num =", G[0], " G den =", G[1])
print("open-loop poles =", np.sort_complex(np.roots(G[1])).round(4))

G num = [4.]  G den = [0.25 1.   0.  ]
open-loop poles = [-4.+0.j  0.+0.j]


In [15]:
Kp = 1.0
C = (np.array([Kp]), np.array([1.0]))         # proportional controller
H = (np.array([0.5]), np.array([1.0]))        # sensor gain H = 0.

fwd = series(C, G)                            # forward path = C·G
T   = feedback(fwd, H)                         # close the loop: G/(1+GH)
print("forward num,den =", fwd[0], fwd[1])
print("closed  num,den =", T[0], T[1])
print("closed-loop poles =", np.sort_complex(np.roots(T[1])).round(4))

forward num,den = [4.] [0.25 1.   0.  ]
closed  num,den = [4.] [0.25 1.   2.  ]
closed-loop poles = [-2.-2.j -2.+2.j]


In [18]:
Hg = 0.5
wn   = np.sqrt(Kp*Km*Hg/tau)
zeta = (1.0/tau) / (2*wn)
Mp   = np.exp(-np.pi*zeta/np.sqrt(1-zeta**2)) * 100   # percent overshoot
tp   = np.pi / (wn*np.sqrt(1-zeta**2))                # peak time
ts   = 4.0 / (zeta*wn)                                # 2% settling (approx)
dc   = np.polyval(T[0], 0) / np.polyval(T[1], 0)      # DC gain T(0)
print(f"wn = {wn:.4f} rad/s,  zeta = {zeta:.4f}")
print(f"Mp = {Mp:.2f} %,  tp = {tp:.4f} s,  ts(2%) approx = {ts:.4f} s")
print(f"DC gain T(0) = {dc:.4f}")

wn = 2.8284 rad/s,  zeta = 0.7071
Mp = 4.32 %,  tp = 1.5708 s,  ts(2%) approx = 2.0000 s
DC gain T(0) = 2.0000


In [20]:
def deriv(x, r):                       # x = [y, y_dot]
    y, yd = x
    return np.array([yd, -2*zeta*wn*yd - wn**2*y + wn**2*dc*r])

dt, Tend, r = 1e-4, 12.0, 1.0
x = np.array([0.0, 0.0]); ymax, tmax = 0.0, 0.0
N = int(Tend/dt); ys = np.empty(N+1)
for k in range(N+1):
    t = k*dt; ys[k] = x[0]
    if x[0] > ymax: ymax, tmax = x[0], t
    k1 = deriv(x, r);            k2 = deriv(x + 0.5*dt*k1, r)
    k3 = deriv(x + 0.5*dt*k2, r); k4 = deriv(x + dt*k3, r)
    x = x + dt/6*(k1 + 2*k2 + 2*k3 + k4)

sim_Mp = (ymax - dc)/dc * 100
sim_ts = next(k*dt for k in range(N, -1, -1) if abs(ys[k]-dc) > 0.02*dc)
print(f"sim overshoot = {sim_Mp:.2f} %  (analytic {Mp:.2f} %)")
print(f"sim peak time = {tmax:.4f} s  (analytic {tp:.4f} s)")
print(f"sim 2% settling = {sim_ts:.4f} s,  final value y(12s) = {ys[-1]:.4f}")

sim overshoot = 4.32 %  (analytic 4.32 %)
sim peak time = 1.5708 s  (analytic 1.5708 s)
sim 2% settling = 2.1080 s,  final value y(12s) = 2.0000


In [24]:
char = [tau, 1.0, Kp*Km*Hg]          # tau*s^2 + 1*s + Kp*Km*H
r = np.roots(char)

print("char poly roots =", np.sort_complex(r).round(4))
print("match:", np.allclose(np.sort_complex(r), np.sort_complex(np.roots(T[1]))))

char poly roots = [-2.-2.j -2.+2.j]
match: True
